# Nemotron Leaderboard Engine

Use this notebook as the Kaggle-side competition notebook for the NVIDIA Nemotron Model Reasoning Challenge.

Before running:
- Attach the competition input.
- Attach the Kaggle model `metric/nemotron-3-nano-30b-a3b-bf16/Transformers/default`.
- Set `Settings -> Accelerator -> GPU RTX Pro 6000` first.
- Keep internet on only long enough to upgrade PyTorch or install model runtime dependencies if the notebook tells you to.

What this notebook now does automatically:
- Creates a unique run folder under `/kaggle/working/artifacts/kaggle_runs/<run_id>/`.
- Logs every major stage to `events.jsonl`.
- Writes structured summaries for environment, data discovery, model load, smoke tests, and submission scaffold generation.
- Builds a zipped run bundle in `/kaggle/working` so each Kaggle attempt is exportable.

Run strategy:
1. Run the environment and input inspection cells.
2. Run the competition/model discovery cells.
3. Run the Blackwell compatibility cell. If it upgrades PyTorch, the kernel will restart automatically.
4. After restart, rerun from the top until you reach the runtime dependency cell.
5. Let the notebook install `mamba-ssm` and `causal-conv1d` if needed.
6. Run the tokenizer cell.
7. Run the full model load cell.
8. If the model loads, run the short generation smoke test.
9. Review the exported run bundle paths at the end and sync the outputs back into the repo loop.

In [ ]:
import json
import os
import platform
import shutil
import socket
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import pandas as pd
import torch

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
COMP_DIR = Path('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge')

NOTEBOOK_NAME = 'nemotron_leaderboard_engine'
RUN_ID = os.environ.get('NEMOTRON_RUN_ID') or f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-{uuid4().hex[:8]}"
RUN_ARTIFACT_ROOT = WORK_ROOT / 'artifacts' / 'kaggle_runs' / RUN_ID
RUN_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
EVENT_LOG_PATH = RUN_ARTIFACT_ROOT / 'events.jsonl'
SESSION_SUMMARY_PATH = RUN_ARTIFACT_ROOT / 'session_summary.json'
SYNC_HINT_PATH = RUN_ARTIFACT_ROOT / 'sync_hint.txt'

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def write_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str), encoding='utf-8')


def write_text(path: Path, payload: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(payload, encoding='utf-8')


def log_event(event_type: str, **payload):
    record = {
        'ts': utc_now_iso(),
        'event_type': event_type,
        'run_id': RUN_ID,
        **payload,
    }
    with EVENT_LOG_PATH.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, sort_keys=True, default=str) + '\n')
    return record


SESSION_STATE = {
    'run_id': RUN_ID,
    'notebook_name': NOTEBOOK_NAME,
    'artifact_root': str(RUN_ARTIFACT_ROOT),
    'event_log_path': str(EVENT_LOG_PATH),
    'started_at': utc_now_iso(),
}


def update_session(**payload):
    SESSION_STATE.update(payload)
    write_json(SESSION_SUMMARY_PATH, SESSION_STATE)


write_text(
    SYNC_HINT_PATH,
    '\n'.join([
        f'Run ID: {RUN_ID}',
        f'Artifact root: {RUN_ARTIFACT_ROOT}',
        f'Event log: {EVENT_LOG_PATH}',
        'Persist this run by saving the Kaggle version or exporting the generated zip bundle from /kaggle/working.',
    ]),
)

print('Run ID:', RUN_ID)
print('Artifact root:', RUN_ARTIFACT_ROOT)
print('Torch version:', torch.__version__)
print('Torch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('Input root exists:', INPUT_ROOT.exists())
print('Working root:', WORK_ROOT)

GPU_NAME = None
GPU_CAPABILITY = None
GPU_VRAM_GB = None
BLACKWELL_GPU = False

if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(f'GPU {index}: {torch.cuda.get_device_name(index)}')
        print('  total_memory_gb =', round(props.total_memory / 1024**3, 2))

    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_CAPABILITY = torch.cuda.get_device_capability(0)
    GPU_VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    BLACKWELL_GPU = 'blackwell' in GPU_NAME.lower() or GPU_CAPABILITY[0] >= 12
    print('GPU capability:', GPU_CAPABILITY)
    print('Blackwell GPU detected:', BLACKWELL_GPU)
else:
    print('No CUDA GPU detected. Stop here and turn on a GPU before loading the model.')

runtime_summary = {
    'torch_version': torch.__version__,
    'torch_cuda_build': torch.version.cuda,
    'cuda_available': bool(torch.cuda.is_available()),
    'gpu_name': GPU_NAME,
    'gpu_capability': GPU_CAPABILITY,
    'gpu_vram_gb': GPU_VRAM_GB,
    'blackwell_gpu': BLACKWELL_GPU,
    'python_version': platform.python_version(),
    'hostname': socket.gethostname(),
    'kaggle_kernel_run_type': os.environ.get('KAGGLE_KERNEL_RUN_TYPE'),
}
write_json(RUN_ARTIFACT_ROOT / 'runtime_summary.json', runtime_summary)
update_session(runtime=runtime_summary)
log_event('session_started', runtime=runtime_summary)

In [ ]:
top_level_inputs = sorted(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
print('Top-level /kaggle/input entries:')
for path in top_level_inputs:
    print('-', path)

sample_submission_candidates = sorted(INPUT_ROOT.rglob('sample_submission.csv'))
test_like_candidates = sorted(
    path for path in INPUT_ROOT.rglob('*')
    if path.name.lower() in {'test.csv', 'test.parquet', 'test.jsonl'}
)
train_like_candidates = sorted(
    path for path in INPUT_ROOT.rglob('*')
    if path.name.lower() in {'train.csv', 'train.parquet', 'train.jsonl'}
)

print('\nSample submission candidates:')
for path in sample_submission_candidates[:10]:
    print('-', path)

print('\nTest file candidates:')
for path in test_like_candidates[:10]:
    print('-', path)

print('\nTrain file candidates:')
for path in train_like_candidates[:10]:
    print('-', path)

input_summary = {
    'top_level_inputs': [str(path) for path in top_level_inputs],
    'sample_submission_candidates': [str(path) for path in sample_submission_candidates[:10]],
    'test_candidates': [str(path) for path in test_like_candidates[:10]],
    'train_candidates': [str(path) for path in train_like_candidates[:10]],
}
write_json(RUN_ARTIFACT_ROOT / 'input_inventory.json', input_summary)
update_session(input_inventory=input_summary)
log_event('inputs_discovered', **input_summary)

In [ ]:
def find_model_candidates(search_root: Path) -> list[Path]:
    candidates = []
    for config_path in search_root.rglob('config.json'):
        parent = config_path.parent
        if (parent / 'tokenizer_config.json').exists():
            candidates.append(parent)

    return sorted(
        set(candidates),
        key=lambda path: ('nemotron' not in str(path).lower(), len(path.parts), str(path)),
    )


MODEL_CANDIDATES = find_model_candidates(INPUT_ROOT)
print('Model candidates:')
for index, path in enumerate(MODEL_CANDIDATES):
    print(index, path)

assert MODEL_CANDIDATES, 'No model root detected under /kaggle/input.'
MODEL_DIR = MODEL_CANDIDATES[0]
print('\nSelected MODEL_DIR =', MODEL_DIR)

SAMPLE_SUBMISSION_PATH = sample_submission_candidates[0] if sample_submission_candidates else None
TEST_DATA_PATH = test_like_candidates[0] if test_like_candidates else None
TRAIN_DATA_PATH = train_like_candidates[0] if train_like_candidates else None
print('SAMPLE_SUBMISSION_PATH =', SAMPLE_SUBMISSION_PATH)
print('TEST_DATA_PATH =', TEST_DATA_PATH)
print('TRAIN_DATA_PATH =', TRAIN_DATA_PATH)

model_selection_summary = {
    'model_candidates': [str(path) for path in MODEL_CANDIDATES],
    'selected_model_dir': str(MODEL_DIR),
    'sample_submission_path': str(SAMPLE_SUBMISSION_PATH) if SAMPLE_SUBMISSION_PATH is not None else None,
    'test_data_path': str(TEST_DATA_PATH) if TEST_DATA_PATH is not None else None,
    'train_data_path': str(TRAIN_DATA_PATH) if TRAIN_DATA_PATH is not None else None,
}
write_json(RUN_ARTIFACT_ROOT / 'model_selection.json', model_selection_summary)
update_session(model_selection=model_selection_summary)
log_event('model_selected', **model_selection_summary)

In [ ]:
print('Competition files:')
competition_files = []
if COMP_DIR.exists():
    for path in sorted(COMP_DIR.rglob('*')):
        if path.is_file():
            competition_files.append(str(path))
            print('-', path)
else:
    print('Competition directory not found:', COMP_DIR)

TEST_COLUMNS = []
TRAIN_COLUMNS = []
test_preview_path = None
train_preview_path = None

if TEST_DATA_PATH is not None:
    test_preview = pd.read_csv(TEST_DATA_PATH, nrows=5)
    TEST_COLUMNS = list(test_preview.columns)
    print('\ntest.csv columns:', TEST_COLUMNS)
    display(test_preview)
    test_preview_path = RUN_ARTIFACT_ROOT / 'test_preview.csv'
    test_preview.to_csv(test_preview_path, index=False)
else:
    print('TEST_DATA_PATH is not available.')

if TRAIN_DATA_PATH is not None:
    train_preview = pd.read_csv(TRAIN_DATA_PATH, nrows=5)
    TRAIN_COLUMNS = list(train_preview.columns)
    print('\ntrain.csv columns:', TRAIN_COLUMNS)
    display(train_preview)
    train_preview_path = RUN_ARTIFACT_ROOT / 'train_preview.csv'
    train_preview.to_csv(train_preview_path, index=False)
else:
    print('TRAIN_DATA_PATH is not available.')

competition_summary = {
    'competition_files': competition_files,
    'test_columns': TEST_COLUMNS,
    'train_columns': TRAIN_COLUMNS,
    'test_preview_path': str(test_preview_path) if test_preview_path is not None else None,
    'train_preview_path': str(train_preview_path) if train_preview_path is not None else None,
}
write_json(RUN_ARTIFACT_ROOT / 'competition_summary.json', competition_summary)
update_session(competition=competition_summary)
log_event('competition_inspected', **competition_summary)

## Blackwell Compatibility Fix

Kaggle's default PyTorch build may not support the Blackwell GPU architecture yet.

This cell will:
- detect that mismatch,
- install a CUDA 12.8-compatible PyTorch build if needed,
- write a marker file into `/kaggle/working`,
- and restart the kernel automatically.

If the kernel restarts, simply rerun from the top.

In [ ]:
import subprocess
import sys

TORCH_UPGRADE_MARKER = WORK_ROOT / '.torch_blackwell_ready'

def parse_cuda_version(raw: str | None) -> tuple[int, int]:
    if not raw:
        return (0, 0)
    parts = raw.split('.')
    major = int(parts[0]) if parts else 0
    minor = int(parts[1]) if len(parts) > 1 else 0
    return (major, minor)


def install_blackwell_torch() -> None:
    attempts = [
        (
            'stable-cu128',
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--quiet',
                '--upgrade',
                'torch',
                'torchvision',
                'torchaudio',
                '--index-url',
                'https://download.pytorch.org/whl/cu128',
            ],
        ),
        (
            'nightly-cu128',
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--quiet',
                '--upgrade',
                '--pre',
                'torch',
                'torchvision',
                'torchaudio',
                '--index-url',
                'https://download.pytorch.org/whl/nightly/cu128',
            ],
        ),
    ]

    last_error = None
    for label, command in attempts:
        print(f'Trying torch install target: {label}')
        log_event('torch_install_attempt', target=label)
        try:
            subprocess.check_call(command)
            log_event('torch_install_succeeded', target=label)
            print(f'Succeeded with {label}')
            return
        except subprocess.CalledProcessError as exc:
            last_error = exc
            log_event('torch_install_failed', target=label, return_code=exc.returncode)
            print(f'Failed with {label}: return code {exc.returncode}')

    raise RuntimeError(
        'Unable to install a Blackwell-compatible torch build. Turn Kaggle internet on, rerun this cell, and only continue once the kernel restarts cleanly.'
    ) from last_error


current_cuda_build = parse_cuda_version(torch.version.cuda)
needs_blackwell_upgrade = bool(torch.cuda.is_available() and BLACKWELL_GPU and current_cuda_build < (12, 8))

print('Current CUDA build:', current_cuda_build)
print('Needs Blackwell torch upgrade:', needs_blackwell_upgrade)
print('Upgrade marker exists:', TORCH_UPGRADE_MARKER.exists())
log_event(
    'torch_compatibility_checked',
    current_cuda_build=current_cuda_build,
    needs_blackwell_upgrade=needs_blackwell_upgrade,
    upgrade_marker_exists=TORCH_UPGRADE_MARKER.exists(),
)

if needs_blackwell_upgrade and not TORCH_UPGRADE_MARKER.exists():
    print('Installing a Blackwell-compatible torch build...')
    install_blackwell_torch()
    TORCH_UPGRADE_MARKER.write_text('upgraded\n', encoding='utf-8')
    update_session(torch_upgrade={'performed': True, 'marker_path': str(TORCH_UPGRADE_MARKER), 'target_cuda': 'cu128'})
    print('Torch upgrade complete. Restarting kernel now. After restart, rerun from the top.')
    log_event('torch_upgrade_completed', marker_path=str(TORCH_UPGRADE_MARKER))
    os.kill(os.getpid(), 9)
elif needs_blackwell_upgrade:
    update_session(torch_upgrade={'performed': True, 'marker_path': str(TORCH_UPGRADE_MARKER), 'target_cuda': 'cu128'})
    log_event('torch_upgrade_reused', marker_path=str(TORCH_UPGRADE_MARKER))
    print('Blackwell upgrade marker found. Continue with the next cells.')
else:
    update_session(torch_upgrade={'performed': False, 'current_cuda_build': current_cuda_build})
    log_event('torch_upgrade_not_required', current_cuda_build=current_cuda_build)
    print('Current torch build is acceptable for this accelerator. Continue with the next cells.')

## Install Nemotron Runtime Dependencies

The Nemotron hybrid model code imports `mamba-ssm` and `causal-conv1d`. Kaggle does not include these by default.

Keep internet on for this step. If installation fails, stop and capture the exact error.

In [ ]:
import importlib.util

RUNTIME_DEPS_MARKER = WORK_ROOT / '.nemotron_runtime_deps_ready'

def collect_runtime_dep_status() -> dict:
    status = {
        'mamba_ssm_spec': importlib.util.find_spec('mamba_ssm') is not None,
        'causal_conv1d_spec': importlib.util.find_spec('causal_conv1d') is not None,
    }

    try:
        import mamba_ssm  # noqa: F401
        from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn  # noqa: F401
        status['mamba_import_ok'] = True
    except Exception as exc:
        status['mamba_import_ok'] = False
        status['mamba_error_type'] = type(exc).__name__
        status['mamba_error_message'] = str(exc)

    try:
        import causal_conv1d  # noqa: F401
        from causal_conv1d import causal_conv1d_fn  # noqa: F401
        status['causal_conv1d_import_ok'] = True
    except Exception as exc:
        status['causal_conv1d_import_ok'] = False
        status['causal_conv1d_error_type'] = type(exc).__name__
        status['causal_conv1d_error_message'] = str(exc)

    status['ready'] = bool(
        status.get('mamba_import_ok') and status.get('causal_conv1d_import_ok')
    )
    return status


runtime_dep_status = collect_runtime_dep_status()
print('Nemotron runtime dependency status:', runtime_dep_status)
write_json(RUN_ARTIFACT_ROOT / 'runtime_dependency_status_before.json', runtime_dep_status)
log_event('runtime_dependencies_checked', status=runtime_dep_status)

if not runtime_dep_status['ready']:
    install_attempts = [
        (
            'combined-no-build-isolation',
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--quiet',
                '--upgrade',
                'ninja',
                'packaging',
                'mamba-ssm[causal-conv1d]',
                '--no-build-isolation',
            ],
        ),
        (
            'causal-conv1d-no-build-isolation',
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--quiet',
                '--upgrade',
                'causal-conv1d>=1.4.0',
                '--no-build-isolation',
            ],
        ),
        (
            'mamba-ssm-no-build-isolation',
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--quiet',
                '--upgrade',
                'mamba-ssm',
                '--no-build-isolation',
            ],
        ),
    ]

    last_error = None
    for label, command in install_attempts:
        print(f'Trying runtime dependency install: {label}')
        log_event('runtime_dependency_install_attempt', target=label)
        try:
            subprocess.check_call(command)
            runtime_dep_status = collect_runtime_dep_status()
            if runtime_dep_status['ready']:
                print(f'Succeeded with {label}')
                log_event('runtime_dependency_install_succeeded', target=label, status=runtime_dep_status)
                break
        except subprocess.CalledProcessError as exc:
            last_error = exc
            print(f'Failed with {label}: return code {exc.returncode}')
            log_event('runtime_dependency_install_failed', target=label, return_code=exc.returncode)
    else:
        error_payload = {
            'stage': 'runtime_dependency_install',
            'status': runtime_dep_status,
        }
        write_json(RUN_ARTIFACT_ROOT / 'runtime_dependency_error.json', error_payload)
        update_session(last_error=error_payload, runtime_dependencies=runtime_dep_status)
        raise RuntimeError(
            'Nemotron runtime dependencies could not be installed. Keep Kaggle internet on and inspect runtime_dependency_error.json in the artifact folder.'
        ) from last_error

write_json(RUN_ARTIFACT_ROOT / 'runtime_dependency_status_after.json', runtime_dep_status)
update_session(runtime_dependencies=runtime_dep_status)
log_event('runtime_dependencies_ready', status=runtime_dep_status)
assert runtime_dep_status['ready'], 'Nemotron runtime dependencies are still missing after install attempts.'
RUNTIME_DEPS_MARKER.write_text('ready\n', encoding='utf-8')
print('Nemotron runtime dependencies are ready.')

## Load The Tokenizer First

This should be cheap. If this fails, the attached model path is wrong or the Kaggle model input is incomplete.

In [ ]:
assert 'MODEL_DIR' in globals(), 'Run the model discovery cell first.'

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), trust_remote_code=True)
tokenizer_summary = {
    'model_dir': str(MODEL_DIR),
    'tokenizer_class': tokenizer.__class__.__name__,
    'vocab_size': getattr(tokenizer, 'vocab_size', 'unknown'),
}
write_json(RUN_ARTIFACT_ROOT / 'tokenizer_summary.json', tokenizer_summary)
update_session(tokenizer=tokenizer_summary)
log_event('tokenizer_loaded', **tokenizer_summary)
print('Tokenizer loaded from:', MODEL_DIR)
print('Tokenizer class:', tokenizer.__class__.__name__)
print('Tokenizer vocab size:', getattr(tokenizer, 'vocab_size', 'unknown'))

## Load The Model

This is the expensive step. If it fails with OOM, stop the session and do not keep retrying blindly.

In [ ]:
assert 'MODEL_DIR' in globals(), 'Run the model discovery cell first.'
assert 'tokenizer' in globals(), 'Run the tokenizer cell first.'
assert RUNTIME_DEPS_MARKER.exists(), 'Run the runtime dependency cell first.'

TORCH_UPGRADE_MARKER = WORK_ROOT / '.torch_blackwell_ready'
current_cuda_build = parse_cuda_version(torch.version.cuda)
assert not (torch.cuda.is_available() and BLACKWELL_GPU and current_cuda_build < (12, 8) and not TORCH_UPGRADE_MARKER.exists()), (
    'Run the Blackwell compatibility cell and let the kernel restart before loading the model.'
)

from transformers import AutoModelForCausalLM

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
model_load_started = time.perf_counter()

try:
    model = AutoModelForCausalLM.from_pretrained(
        str(MODEL_DIR),
        torch_dtype=dtype,
        trust_remote_code=True,
        device_map='auto',
        low_cpu_mem_usage=True,
    )
    model.eval()
except Exception as exc:
    error_payload = {
        'stage': 'model_load',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback': traceback.format_exc(),
    }
    write_json(RUN_ARTIFACT_ROOT / 'model_load_error.json', error_payload)
    update_session(last_error=error_payload, model_load={'status': 'failed'})
    log_event('model_load_failed', error_type=error_payload['error_type'], error_message=error_payload['error_message'])
    raise

def model_input_device(model):
    if hasattr(model, 'device'):
        return model.device
    return next(iter(model.parameters())).device


model_load_summary = {
    'status': 'loaded',
    'dtype': str(dtype),
    'input_device': str(model_input_device(model)),
    'device_map': getattr(model, 'hf_device_map', 'single-device'),
    'latency_ms': int((time.perf_counter() - model_load_started) * 1000),
}
write_json(RUN_ARTIFACT_ROOT / 'model_load.json', model_load_summary)
update_session(model_load=model_load_summary)
log_event('model_loaded', **model_load_summary)
print('Model loaded successfully.')
print('Input device:', model_input_device(model))
print('Device map:', getattr(model, 'hf_device_map', 'single-device'))

In [ ]:
assert 'model' in globals(), 'Run the full model load cell first.'

def tokenize_messages(messages, enable_thinking=None):
    kwargs = {
        'tokenize': True,
        'add_generation_prompt': True,
        'return_tensors': 'pt',
    }
    if enable_thinking is not None:
        try:
            return tokenizer.apply_chat_template(messages, enable_thinking=enable_thinking, **kwargs)
        except TypeError:
            print('Tokenizer does not expose enable_thinking; retrying without it.')
    return tokenizer.apply_chat_template(messages, **kwargs)


messages = [
    {
        'role': 'user',
        'content': 'Solve briefly: what is 17 * 19? Return only the final answer.',
    }
]

inputs = tokenize_messages(messages, enable_thinking=False).to(model_input_device(model))
generation_started = time.perf_counter()

with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_new_tokens=32,
        do_sample=False,
        num_beams=1,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_tokens = outputs[0][inputs.shape[-1]:]
generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
smoke_test_summary = {
    'mode': 'no_think',
    'prompt': messages[0]['content'],
    'generated_text': generated_text,
    'latency_ms': int((time.perf_counter() - generation_started) * 1000),
    'max_new_tokens': 32,
}
write_json(RUN_ARTIFACT_ROOT / 'smoke_test_no_think.json', smoke_test_summary)
update_session(smoke_test_no_think=smoke_test_summary)
log_event('smoke_test_completed', **smoke_test_summary)
print(generated_text)

In [ ]:
assert 'model' in globals(), 'Run the full model load cell first.'

RUN_THINKING_TEST = False

if RUN_THINKING_TEST:
    thinking_started = time.perf_counter()
    thinking_inputs = tokenize_messages(messages, enable_thinking=True).to(model_input_device(model))
    with torch.no_grad():
        thinking_outputs = model.generate(
            thinking_inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=1.0,
            top_p=1.0,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_tokens = thinking_outputs[0][thinking_inputs.shape[-1]:]
    thinking_text = tokenizer.decode(generated_tokens, skip_special_tokens=False)
    thinking_summary = {
        'mode': 'think',
        'prompt': messages[0]['content'],
        'generated_text': thinking_text,
        'latency_ms': int((time.perf_counter() - thinking_started) * 1000),
        'max_new_tokens': 128,
    }
    write_json(RUN_ARTIFACT_ROOT / 'smoke_test_think.json', thinking_summary)
    update_session(smoke_test_think=thinking_summary)
    log_event('thinking_test_completed', **thinking_summary)
    print(thinking_text)
else:
    log_event('thinking_test_skipped')
    print('Set RUN_THINKING_TEST = True only after the short no-thinking smoke test succeeds.')

In [ ]:
submission_summary = {
    'used_sample_submission': False,
    'submission_path': None,
    'artifact_submission_path': None,
    'columns': None,
}

if SAMPLE_SUBMISSION_PATH is not None:
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    print('Sample submission shape:', sample_submission.shape)
    print('Columns:', list(sample_submission.columns))
    display(sample_submission.head())

    prediction_columns = [
        column for column in sample_submission.columns
        if column.lower() not in {'id', 'index'}
    ]

    if prediction_columns:
        submission = sample_submission.copy()
        submission[prediction_columns[0]] = ''
        submission_path = WORK_ROOT / 'submission.csv'
        artifact_submission_path = RUN_ARTIFACT_ROOT / 'submission.csv'
        submission.to_csv(submission_path, index=False)
        submission.to_csv(artifact_submission_path, index=False)
        submission_summary.update({
            'used_sample_submission': True,
            'submission_path': str(submission_path),
            'artifact_submission_path': str(artifact_submission_path),
            'columns': list(submission.columns),
        })
        print('Draft submission scaffold written to:', submission_path)
        display(submission.head())
    else:
        print('Could not infer the prediction column from the sample submission.')
elif TEST_DATA_PATH is not None and TEST_COLUMNS:
    test_id_col = next((column for column in TEST_COLUMNS if column.lower() in {'id', 'index'}), TEST_COLUMNS[0])
    test_ids = pd.read_csv(TEST_DATA_PATH, usecols=[test_id_col])
    placeholder_submission = pd.DataFrame({test_id_col: test_ids[test_id_col], 'answer': ''})
    placeholder_path = WORK_ROOT / 'submission_placeholder.csv'
    artifact_placeholder_path = RUN_ARTIFACT_ROOT / 'submission_placeholder.csv'
    placeholder_submission.to_csv(placeholder_path, index=False)
    placeholder_submission.to_csv(artifact_placeholder_path, index=False)
    submission_summary.update({
        'used_sample_submission': False,
        'submission_path': str(placeholder_path),
        'artifact_submission_path': str(artifact_placeholder_path),
        'columns': list(placeholder_submission.columns),
    })
    print('No sample submission was attached. Wrote a provisional placeholder to:', placeholder_path)
    display(placeholder_submission.head())
    print('Replace the provisional answer column name after the official submission schema is confirmed.')
else:
    print('Neither sample submission nor test columns are available.')

write_json(RUN_ARTIFACT_ROOT / 'submission_summary.json', submission_summary)
update_session(submission=submission_summary, completed_at=utc_now_iso())
log_event('submission_scaffold_written', **submission_summary)

bundle_base = WORK_ROOT / f'kaggle_run_{RUN_ID}'
bundle_path = Path(shutil.make_archive(str(bundle_base), 'zip', root_dir=RUN_ARTIFACT_ROOT.parent, base_dir=RUN_ARTIFACT_ROOT.name))
update_session(export_bundle_path=str(bundle_path))
log_event('run_bundle_created', bundle_path=str(bundle_path))
print('\nRun artifacts ready:')
print('-', RUN_ARTIFACT_ROOT)
print('-', EVENT_LOG_PATH)
print('-', SESSION_SUMMARY_PATH)
print('-', bundle_path)

## Next Step

If the model loads cleanly and the smoke test works on the selected GPU, replace the toy prompt cell with the real competition inference loop and build predictions from the competition test input.

Every run now leaves behind:
- `events.jsonl` for stage-by-stage logging,
- `session_summary.json` for the latest structured state,
- preview files and scaffolded submission artifacts,
- and a zip bundle in `/kaggle/working` for export or repo sync.